# EDA on metadata

Controlli statistici di base sui due output del modulo di retrieval:
- i CSV di `scripts/data_summary.py` (presenza/assenza per `(object, space, modality)`)
- i `participants.tsv` copiati da EBRAIN per ciascun dataset

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path("/home/etosato/Projects/NEMESIS_fs")
DATASETS = ["UNIPD/WashU", "UNIPD/PASPORT", "UNIPD/PSP", "UKLFR/stroke_UKLFR"]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Controllo report CSV

In [ ]:
def _data_summary_path(dataset: str) -> Path:
    safe_name = dataset.replace("/", "_")
    return PROJECT_ROOT / "reports" / "data_retrieval" / "clinical_connectome" / f"data_summary__{safe_name}.csv"


# one DataFrame per dataset, indexed by subject
data_summary: dict[str, pd.DataFrame] = {}
for dataset in DATASETS:
    path = _data_summary_path(dataset)
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found - run "
            "`python scripts/data_summary.py --config config/retrieval.json` first"
        )
    data_summary[dataset] = pd.read_csv(path, index_col="subject")

for dataset, df in data_summary.items():
    print(f"{dataset}: {df.shape[0]} subjects, {df.shape[1]} (object/space/modality) columns")

Sanity check: same registry columns everywhere

In [ ]:
# every CSV comes from the same file_patterns.json registry, so all datasets
# should expose the exact same set of (object, space, modality) columns
column_sets = {dataset: set(df.columns) for dataset, df in data_summary.items()}
reference_columns = next(iter(column_sets.values()))
for dataset, columns in column_sets.items():
    if columns != reference_columns:
        raise ValueError(f"{dataset} has a different column set than the others: {columns ^ reference_columns}")
print("all datasets expose the same (object, space, modality) columns, as expected from a shared registry")


Per-dataset presence/absence stats

In [ ]:
def presence_stats(df: pd.DataFrame) -> pd.DataFrame:
    """Per-column count/percentage of subjects with a real file ("present")."""
    n_present = df.eq("present").sum()
    n_subjects = len(df)
    return pd.DataFrame(
        {
            "n_present": n_present,
            "n_missing": n_subjects - n_present,
            "pct_present": (n_present / n_subjects * 100).round(1),
        }
    )


for dataset, df in data_summary.items():
    print(f"\n=== {dataset} ({len(df)} subjects) ===")
    display(presence_stats(df))


Cross-dataset completeness pivot

In [ ]:
# long format (one row per dataset x column) so datasets can be compared side by side
presence_long = pd.concat(
    [
        presence_stats(df).reset_index(names="column").assign(dataset=dataset)
        for dataset, df in data_summary.items()
    ],
    ignore_index=True,
)

# rows = dataset, columns = object/space/modality, values = % of subjects with the file
completeness_pivot = presence_long.pivot(index="dataset", columns="column", values="pct_present")
display(completeness_pivot)

Per-subject completeness (flag subjects with nothing at all)

In [ ]:
# how many of the registered (object, space, modality) combinations does each subject have?
for dataset, df in data_summary.items():
    n_present_per_subject = df.eq("present").sum(axis=1)
    orphans = n_present_per_subject[n_present_per_subject == 0]
    print(f"{dataset}: {len(orphans)} subject(s) with NOTHING present across all registered combinations")
    if len(orphans) > 0:
        print(f"  {orphans.index.tolist()}")


## Controllo TSV

In [ ]:
def _participants_path(dataset: str) -> Path:
    return PROJECT_ROOT / "data" / "clinical_connectome" / dataset / "participants.tsv"


participants: dict[str, pd.DataFrame] = {}
for dataset in DATASETS:
    path = _participants_path(dataset)
    if not path.exists():
        raise FileNotFoundError(f"{path} not found - has the retrieval pipeline been run for this dataset?")
    # "n/a" is this project's missing-value marker; empty cells (seen in the
    # UKLFR sheet) are already NaN under pandas' default na_values
    participants[dataset] = pd.read_csv(path, sep="\t", na_values=["n/a"])

for dataset, df in participants.items():
    print(f"{dataset}: {df.shape[0]} participants, {df.shape[1]} columns")


duplicate participant_id (within and across datasets)

In [ ]:
for dataset, df in participants.items():
    duplicated = df["participant_id"][df["participant_id"].duplicated()]
    status = "no duplicates" if duplicated.empty else f"DUPLICATES: {duplicated.tolist()}"
    print(f"{dataset}: {status}")

# participant_id is expected to be globally unique across all four datasets
all_ids = pd.concat([df["participant_id"] for df in participants.values()], ignore_index=True)
cross_dataset_duplicates = all_ids[all_ids.duplicated()]
print(f"\ncross-dataset duplicate participant_id: {cross_dataset_duplicates.tolist() or 'none'}")


Consistency check: does the file's own dataset column match where we found it?

In [ ]:
# each participants.tsv carries its own "dataset" column - it should match the
# last path component we used to locate the file, otherwise something is
# mislabeled at the source
for dataset, df in participants.items():
    expected = dataset.split("/")[-1]
    found = set(df["dataset"].dropna().unique())
    if found != {expected}:
        print(f"MISMATCH for {dataset}: 'dataset' column has {found}, expected {{{expected!r}}}")
    else:
        print(f"{dataset}: 'dataset' column consistent ({expected!r})")


Missingness per column

In [ ]:
def missingness_stats(df: pd.DataFrame) -> pd.DataFrame:
    n_missing = df.isna().sum()
    return pd.DataFrame(
        {"n_missing": n_missing, "pct_missing": (n_missing / len(df) * 100).round(1)}
    ).sort_values("pct_missing", ascending=False)


for dataset, df in participants.items():
    print(f"\n=== {dataset} ===")
    display(missingness_stats(df))

Numeric summary

In [ ]:
for dataset, df in participants.items():
    print(f"\n=== {dataset} - numeric columns ===")
    display(df.select_dtypes(include="number").describe().T)


Categorical breakdown (guarded: schema differs per dataset)

In [ ]:
CATEGORICAL_COLUMNS = ["sex", "disease_id", "lesion_side", "handedness"]

for dataset, df in participants.items():
    print(f"\n=== {dataset} ===")
    for col in CATEGORICAL_COLUMNS:
        if col not in df.columns:
            print(f"  ({col} not in this dataset's schema)")
            continue
        print(f"  {col}:")
        print(df[col].value_counts(dropna=False).to_string())

Combined view on shared "core" columns

In [ ]:
CORE_COLUMNS = ["participant_id", "dataset", "center_id", "disease_id", "age", "sex"]

combined = pd.concat([df[CORE_COLUMNS] for df in participants.values()], ignore_index=True)
print(f"combined: {combined.shape[0]} participants across {combined['dataset'].nunique()} datasets")

display(combined.groupby("dataset")["age"].describe())
display(combined.groupby(["dataset", "sex"]).size().unstack(fill_value=0))
